# Baselines

Two reference "models" that every real model has to beat:

1. **Constant**: always predicts the mean `cnt` of the training part.
2. **Hour x day type**: predicts the mean `cnt` of the training part for the
   given pair (`hr`, `workingday`).

Both are fitted **inside each fold** of the validation scheme (via
`evaluate`), so the means never include validation data (no leakage).

In [1]:
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.dummy import DummyRegressor

from src.data import load_data
from src.validation import SEED, SCHEMES, evaluate

X, y = load_data()
print(X.shape, y.shape)

(17379, 9) (17379,)


## Hour x day type baseline

A tiny sklearn-compatible estimator. It inherits from `BaseEstimator`, so
`clone` (used in `evaluate`) works. Pairs that were not seen in the training
part fall back to the training mean.

In [2]:
class HourDayBaseline(RegressorMixin, BaseEstimator):
    """Mean of the target for each (hr, workingday) pair of the training part."""

    def fit(self, X, y):
        d = X[["hr", "workingday"]].copy()
        d["y"] = y.to_numpy()
        self.table_ = d.groupby(["hr", "workingday"])["y"].mean()
        self.global_mean_ = float(d["y"].mean())
        return self

    def predict(self, X):
        keys = pd.MultiIndex.from_frame(X[["hr", "workingday"]])
        return self.table_.reindex(keys).fillna(self.global_mean_).to_numpy()

## Evaluation in both schemes

In [3]:
baselines = {
    "baseline_const": DummyRegressor(strategy="mean"),
    "baseline_hour_day": HourDayBaseline(),
}

summary = []
for name, model in baselines.items():
    for scheme in SCHEMES:
        folds, preds = evaluate(model, X, y, scheme, name)
        summary.append({
            "model": name,
            "scheme": scheme,
            "rmse_mean": folds["rmse"].mean(),
            "rmse_std": folds["rmse"].std(),
            "mae_mean": folds["mae"].mean(),
            "r2_mean": folds["r2"].mean(),
        })

pd.DataFrame(summary).round(3)

,model,scheme,rmse_mean,rmse_std,mae_mean,r2_mean
0,baseline_const,random,181.372,3.540,142.415,-0.001
1,baseline_const,blocked,182.116,47.228,147.169,-0.280
2,baseline_const,chrono,215.611,52.739,165.455,-0.159
3,baseline_hour_day,random,109.942,1.384,72.959,0.632
4,baseline_hour_day,blocked,119.111,38.852,83.458,0.304
5,baseline_hour_day,chrono,133.292,44.825,95.482,0.562


## Per-fold results (chronological scheme)

The folds cover different periods, so their errors are not interchangeable.

In [4]:
pd.read_csv("results/baseline_hour_day_chrono_folds.csv")

,fold,n_train,n_test,test_start,test_end,rmse,mae,r2,rmsle
0,1,8657,1746,2012-01-01 12:00:00,2012-03-14 13:00:00,65.677142,42.638147,0.768854,0.479683
1,2,10403,1753,2012-03-14 14:00:00,2012-05-26 16:00:00,159.678309,115.335148,0.430587,0.626170
2,3,12156,1754,2012-05-26 17:00:00,2012-08-07 18:00:00,162.414163,122.515410,0.449770,0.607197
3,4,13910,1754,2012-08-07 19:00:00,2012-10-19 20:00:00,169.740152,125.598505,0.496893,0.558875
4,5,15664,1715,2012-10-19 21:00:00,2012-12-31 23:00:00,108.950987,71.324968,0.664964,0.628595


## Conclusions

_To be written after looking at the numbers._